# Dataset Pre treatment

I will be creating a tokenizer for the dataset along with splitting this data into training, test and evaluation dataset

### Dataset inspection

I will be looking at random samples in the dataset to find if the dataset mapping is correct

In [1]:
englishTrainFile = 'train.en'
malayalamTrainFile = 'train.ml'

In [2]:
with open(englishTrainFile, 'r') as f:
    englishLineCount = sum(1 for line in f)
print(englishLineCount)

5924426


In [3]:
with open(malayalamTrainFile, 'r') as f:
    malayalamLineCount = sum(1 for line in f)
print(malayalamLineCount)

5924426


In [4]:
malayalamLineCount == englishLineCount

True

In [3]:
import numpy as np
import random

In [6]:
englishSample = []
malaylamSample = []
sampleIndices = []

with open(malayalamTrainFile, 'r') as f:
    for i, line in enumerate(f):
        if len(sampleIndices) >= 100:
            break

        if random.randint(1, 750) == 69:
            sampleIndices.append(i)
            malaylamSample.append(line)

j = 0
with open(englishTrainFile, 'r') as f:
    for i, line in enumerate(f):
        if len(englishSample) >= 100:
            break
        if i == sampleIndices[j]:
            englishSample.append(line)
            j += 1

In [7]:
print(len(malaylamSample), len(englishSample))

100 100


In [8]:
sampleDataset = tuple(zip(englishSample, malaylamSample))
for eng, mal in sampleDataset:
    print(eng.replace('\n', ''), mal.replace('\n', ''))

The fact that Jason spent his time immersed in personal study when he was well, and thus had a spiritual reserve to draw upon when he needed it, made me think. ഇസഡ്‌. ജി., പരാഗ്വേ
Manipur has a 60-member Assembly. 60 അംഗ നിയമസഭയാണ് മണിപ്പൂരില്‍.
Pull over. ഒതുക്കി നിറുത്തൂ.
The phone runs the Qualcomm Snapdragon 845 processor. ക്വാല്‍കോം സ്‌നാപ്ഡ്രാഗണ്‍ 845 പ്രൊസസറിലാണ് ഫോണ്‍ പ്രവര്‍ത്തിക്കുന്നത്.
He is under observation at the Medical College Hospital. മെഡിക്കൽ കോളേജ് ആശുപത്രിയിൽ ഐസൊലേഷനിൽ കഴിയുകയാണ്.
Pinarayi's Facebook post: സന്ദീപാനന്ദഗിരിയുടെ ഫേസ്ബുക്ക് പോസ്റ്റ്:
Kerala doesnt fall on the path of Hikka cyclone but the disaster management authority has warned fishermen to be alert while venturing into Arabian sea. കേരളം ‘ഹികാ’ ചുഴലിക്കാറ്റിന്റെ സഞ്ചാരപഥത്തിൽ ഇല്ലെങ്കിലും അറബിക്കടലിൽ മീൻപിടിക്കാൻ പോകുന്നവർ ജാഗ്രത പാലിക്കണമെന്ന് ദുരന്തനിവാരണ അതോറിറ്റി മുന്നറിയിപ്പ് നൽകി.
he asked with a laugh. ചിരിച്ചുകൊണ്ട് ഊർമ്മിള ചോദിച്ചു.
This is the idea. ഇതാണ് ആശയം.
Press %1 while NumLock is a

Dataset mappings look fine in all the samples i inspected

### Creating tokenizer

In [9]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

c:\Users\ivanr\.conda\envs\pytorchEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
engTokenizer = Tokenizer(BPE(unk_token='<unk>'))
malTokenizer = Tokenizer(BPE(unk_token='<unk>'))
engTokenizer.pre_tokenizer = Whitespace()
malTokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=4000, special_tokens=['<bos>', '<eos>', '<pad>', '<unk>'])

engFiles = ['train.en']
malFiles = ['train.ml']

engTokenizer.train(engFiles, trainer)
malTokenizer.train(malFiles, trainer)

engTokenizer.save('engTokenizer.json')
malTokenizer.save('malTokenizer.json')

In [11]:
engTok = PreTrainedTokenizerFast(tokenizer_file='engTokenizer.json')
malTok = PreTrainedTokenizerFast(tokenizer_file='malTokenizer.json')

In [12]:
print(engTok('Hello world!')['input_ids'])
print(malTok('നിങ്ങൾക്ക് സമാധാനം ഉണ്ടാകട്ടെ')['input_ids'])

[200, 216, 81, 664, 4]
[3189, 2506, 3807, 2676, 974, 3225, 990, 3267]


### Deciding the length of sequence and embedding

In [13]:
engLens = []
malLens = []
with open(englishTrainFile, 'r') as f:
    for i, line in enumerate(f):
        engLens.append(len(engTok(line.replace('\n', ''))['input_ids']))
        if i % 10000 == 0:
            print(f"[{i}/{englishLineCount}] English ({round(100 * i / englishLineCount, 3)}%)", end="\r")

with open(malayalamTrainFile, 'r') as f:
    for i, line in enumerate(f):
        malLens.append(len(malTok(line.replace('\n', ''))['input_ids']))
        if i % 10000 == 0:
            print(f"[{i}/{malayalamLineCount}] Malayalam ({round(100 * i / malayalamLineCount, 3)}%)", end="\r")

engLens = np.array(engLens)
malLens = np.array(malLens)

In [14]:
print(np.percentile(engLens, [85, 90, 95, 99, 99.99, 99.999]))
print(np.percentile(malLens, [85, 90, 95, 99, 99.99, 99.999]))

[ 27.       33.       42.       67.      201.      353.75575]
[ 34.       41.       53.       88.      289.      565.75575]


In [15]:
# 64 is decided as the sequence length

In [16]:
shouldInclude = np.logical_and((engLens <= 64), (malLens <= 64))

In [24]:
100 * shouldInclude.astype(int).sum() / len(shouldInclude), shouldInclude.astype(int).sum()
# almost 3 percentage of datapoints are dropped 

(np.float64(97.02702000160015), np.int64(5748294))

In [29]:
trainRatio = 0.85
valRatio = 0.1
testRatio = 0.05

In [ ]:
with open(malayalamTrainFile, 'r') as mal, \
    open(englishTrainFile, 'r') as eng, \
    open('malTrain.txt', 'w') as malTrain, \
    open('engTrain.txt', 'w') as engTrain, \
    open('malVal.txt', 'w') as malVal, \
    open('engVal.txt', 'w') as engVal, \
    open('malTest.txt', 'w') as malTest, \
    open('engTest.txt', 'w') as engTest:
        
        for i, (malLine, engLine) in enumerate(zip(mal, eng)):
                if not shouldInclude[i]:
                        continue
                
                probability = random.random()
                if probability <= trainRatio:
                        malTrain.write(malLine)
                        engTrain.write(engLine)
                elif probability <= trainRatio + valRatio:
                        malVal.write(malLine)
                        engVal.write(engLine)
                else:
                        malTest.write(malLine)
                        engTest.write(engLine)

                if i % 10000:
                        print(f"[{i}/{malayalamLineCount}] Progress ({round(100 * i / malayalamLineCount, 3)}%)", end="\r")

                

### Vereifying the data split

In [5]:
englishTrainSample = []
malaylamTrainSample = []
trainSampleIndices = []

malayalamTrainFile = 'malTrain.txt'
englishTrainFile = 'engTrain.txt'

with open(malayalamTrainFile, 'r') as f:
    for i, line in enumerate(f):
        if len(trainSampleIndices) >= 100:
            break

        if random.randint(1, 750) == 69:
            trainSampleIndices.append(i)
            malaylamTrainSample.append(line)

j = 0
with open(englishTrainFile, 'r') as f:
    for i, line in enumerate(f):
        if len(englishTrainSample) >= 100:
            break
        if i == trainSampleIndices[j]:
            englishTrainSample.append(line)
            j += 1

assert len(englishTrainSample) == len(malaylamTrainSample)

sampleDataset = tuple(zip(englishTrainSample, malaylamTrainSample))
for eng, mal in sampleDataset:
    print(eng.replace('\n', ''), mal.replace('\n', ''))

The Union Minister... കേന്ദ്രമന്ത്രിക്ക് കിട്ടുന്ന .
Woman found dead in in-laws house യുവതിയെ ഭർതൃവീട്ടിൽ കഴുത്തറുത്ത് മരിച്ച നിലയിൽ കണ്ടെത്തി
My son was studying BA. എന്റെ മകളുടെ മകന്‍ ബിബിഎക്ക്‌ പഠിക്കുന്നു.
It is unknown when the video was filmed. വിഡിയോ ചിത്രീകരിച്ചത് എപ്പോഴാണെന്ന് വ്യക്തമല്ല.
They have no other thing apart from being dishonest. അവരില്‍ സ്വാര്‍ഥതയല്ലാതെ മറ്റൊന്നുമില്ല.
His sad demise is major loss to the community. സമൂഹത്തിനുണ്ടായ കനത്ത നഷ്ടമാണ് സഖാവിന്റെ മരണം.
A simple process and system monitor. ഒരു ലളിതമായ പ്രക്രിയയുടേയും സിസ്റ്റത്തിന്റേയും നിരീക്ഷകന്‍.
Now it feels hugely valuable. വളരെ അപൂർവ്വമായതിനാൽ വലിയ വിലയേറിയതാണ് ഇവ.
He was immediately detained by the police. അദ്ദേഹത്തെ ഉടന്‍ പോലീസ്‌ അറസ്‌റ്റ്‌ ചെയ്‌തു നീക്കി.
100 per child. 100 കുട്ടികള്‍.
The actor was snapped while shooting for the film. ഷൂട്ടിംഗിനിടയില്‍ ആണ് നടന്‍ വിദ്യയെ അടിച്ചത്.
Rajnath Singh said. ” രാജ്‌നാഥ് സിങ് പറഞ്ഞു.
We are a religious nation. മതസഹിതമായ മതേതരത്വമാണ് നമ്മുടേത്.
The counting

In [13]:
englishValSample = []
malayalamValSample = []

malValFile = 'malVal.txt'
engValFile = 'engVal.txt'

with open(malValFile, 'r') as f:
    for i, line in enumerate(f):
        if len(malayalamValSample) >= 100:
            break
        malayalamValSample.append(line)

j = 0
with open(engValFile, 'r') as f:
    for i, line in enumerate(f):
        if len(englishValSample) >= 100:
            break
        englishValSample.append(line)


sampleDataset = tuple(zip(englishValSample, malayalamValSample))
for eng, mal in sampleDataset:
    print(eng.replace('\n', ''), mal.replace('\n', ''))

It was the first time something like that was happening in my life. തന്റെ ജീവിതത്തില്‍ ആദ്യമായിയായിരുന്നു ഇങ്ങനെ ഒരു സംഭവം നടന്നത്.
Which way will you go? ഏത് വഴിക്കാണ് കൊണ്ടു പോകുന്നത്?
The police said a case under the Excise Act had been registered against him. ഇയാൾക്കെതിരെ ജാമ്യമില്ലാ വകുപ്പ് പ്രകാരമാണ് കേസ് എടുത്തിരിക്കുന്നതെന്നും പൊലീസ് പറയുന്നു.
I cannot say that to them. നാണം കൊണ്ട് അത് അവരോട് പറയാൻ എനിയ്ക്ക് കഴിഞ്ഞില്ല.
'Are we not Indians? ''ഇന്ത്യാക്കാരാണല്ലൊ, അല്ലേ?
The States include Gujarat, Maharashtra, Goa, Karnataka, Kerala and Tamil Nadu. ബാക്കി കര്‍ണാടക, തമിഴ്‌നാട്, ഗോവ, ആന്ധ്ര, ഗുജറാത്ത്, മഹാരാഷ്ട്ര, ഒഡിഷ സംസ്ഥാനങ്ങളില്‍നിന്നാണ് എത്തിക്കുന്നത്.
There are other such cases. ഇത്തരം സംഭവങ്ങള്‍ വേറെയുമുണ്ട്.
Everybody will be happy and content. എല്ലാവർക്കും രസകരവും രസകരമായ ഉണ്ടാകും.
The film is also nominated for the best film, actor, director and script. മികച്ച സംവിധായകന്‍, നടന്‍, സഹ നടന്‍, സഹ നടി, തിരക്കഥ തുടങ്ങിയവയ്ക്കും ചിത്രം നിര്‍ദേശിക്കപ്പെട്ടിട്ടുണ്ട്.
Music is b

In [15]:
malaylamTestSample = []
englishTestSample = []

malTestFile = 'malTest.txt'
engTestFile = 'engTest.txt'

with open(malTestFile, 'r') as f:
    for i, line in enumerate(f):
        if len(malaylamTestSample) >= 100:
            break
        malaylamTestSample.append(line)

j = 0
with open(engTestFile, 'r') as f:
    for i, line in enumerate(f):
        if len(englishTestSample) >= 100:
            break
        englishTestSample.append(line)


sampleDataset = tuple(zip(englishTestSample, malaylamTestSample))
for eng, mal in sampleDataset:
    print(eng.replace('\n', ''), mal.replace('\n', ''))

The same as on the stone. കല്ലിൽ കണ്ട അതേ രൂപം.
To that end, we can reflect on some past examples of courage. നമുക്ക് ഇപ്പോൾ, ധൈര്യത്തിന്‍റെ ഉത്തമമാതൃകകളായ ചിലരെക്കുറിച്ച് ചിന്തിക്കാം.
But I truly understand nothing more matters than ur health n well being. പക്ഷെ സ്വന്തം ആരോഗ്യത്തെക്കാള്‍ വലുതായി മറ്റൊന്നുമില്ലെന്ന് ഞാന്‍ മനസിലാക്കുന്നു.
Trains partially cancelled പാസ്സഞ്ചര്‍ ട്രെയിനുകള്‍ ഭാഗികമായി റദ്ദു ചെയ്തു
Chances of victory വിജയിക്കുന്നു സാധ്യത
But the Forest Department till date has not taken any measure to solve the problem. എന്നാല്‍ വന്യമൃഗശല്യത്തിന് പരിഹാരം ഉണ്ടാക്കാന്‍ വനം വകുപ്പ് അധികൃതര്‍ക്ക് ഇതുവരെ കഴിഞ്ഞിട്ടില്ല.
However, the truth is yet to be discovered. എന്തായാലും ഇതിന്റെ സത്യാവസ്ഥ ഇതുവരെ പുറത്തുവന്നിട്ടില്ല.
The two framework agreements were signed at the White House and were witnessed by President Jimmy Carter. രണ്ട് കരാറുകളുടേയും ചട്ടക്കൂടുകൾ വൈറ്റ് ഹൌസിൽ അക്കാലത്തെ അമേരിക്കൻ പ്രസിഡന്റ് ജിമ്മി കാർട്ടറുടെ സാന്നിദ്ധ്യത്തിലാണ് ഒപ്പുവയ്ക്കപ്പെട്ടത്.
However, she ended 